In [2]:
import cv2
import mediapipe as mp
import os
import matplotlib.pyplot as plt 
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.signal import savgol_filter
import pywt
from ultralytics import YOLO
from scipy.ndimage import zoom

def process_video(input_video_path, output_base_dir):
    import mediapipe as mp
    # Função para criar o filtro passa-banda
    def butter_bandpass(lowcut, highcut, fs, order=4):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = butter(order, [low, high], btype='band')
        return b, a

    # Aplicação do filtro passa-banda
    def apply_bandpass_filter(data, lowcut, highcut, fs, order=4):
        b, a = butter_bandpass(lowcut, highcut, fs, order=order)
        filtered_data = filtfilt(b, a, data)
        return filtered_data

    def moving_average(signal, window_size):
        return np.convolve(signal, np.ones(window_size) / window_size, mode='valid')

    # Remoção de baseline com ajuste de spline
    def remove_baseline(signal, poly_order=2):
        x = np.arange(len(signal))
        coeffs = np.polyfit(x, signal, poly_order)  
        baseline = np.polyval(coeffs, x)     
        return signal - baseline  # Remove a baseline

    def normalized_signal(signal):
        sig= np.array(signal) 
        signal_min = np.min(sig)
        signal_max = np.max(sig)
        normalized_signal = (sig - signal_min) / (signal_max - signal_min)
        return normalized_signal

    model = YOLO("C:/Users/Douglas/Downloads/yolov8m_200e.pt")

# Inicializar FaceMesh
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)

    # Vídeo de entrada
    cap = cv2.VideoCapture(input_video_path) 
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # ROI e parâmetros
    resize_dim = (115, 50)      
    amplification_factor_x = 0.7
    amplification_factor_y = 0.7  

    pontos_referencia = [103, 67, 104, 109, 10, 338, 297, 332, 333, 299, 337, 151, 108, 69, 104, 105, 66, 107, 9, 336, 296, 334]
    conexoes = [
        (103, 67), (67, 109), (109, 10), (10, 338),
        (338, 297), (297, 332), (103, 105),
        (105, 66), (66, 107), (107, 9), (9, 336), (336, 296), (296, 334), (334, 332)
    ]

    # Sinais
    green_signal, R_channel, G_channel, B_channel = [], [], [], []
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, device='cuda')
        boxes = results[0].boxes.xyxy.cpu().numpy()

        if len(boxes) > 0:
            # Pegamos a primeira detecção
            x1, y1, x2, y2 = boxes[0].astype(int)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            face_roi = frame[y1:y2, x1:x2]
            frame_rgb_roi = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)

            result = face_mesh.process(frame_rgb_roi)

            if result.multi_face_landmarks:
                for face_landmarks in result.multi_face_landmarks:
                    pontos = []
                    for idx in pontos_referencia:
                        lm = face_landmarks.landmark[idx]
                        x = int(lm.x * (x2 - x1)) + x1
                        y = int(lm.y * (y2 - y1)) + y1
                        pontos.append((x, y))

                    if pontos:
                        xs, ys = [p[0] for p in pontos], [p[1] for p in pontos]
                        x_min, x_max = max(0, min(xs)), min(frame_width, max(xs))
                        y_min, y_max = max(0, min(ys)), min(frame_height, max(ys))

                        roi = frame[y_min:y_max, x_min:x_max]
                        if roi.size > 0:
                            resized_roi = cv2.resize(roi, resize_dim)

                            R_channel.append(np.mean(resized_roi[:, :, 2]))
                            G_channel.append(np.mean(resized_roi[:, :, 1]))
                            B_channel.append(np.mean(resized_roi[:, :, 0]))
                            green_signal.append(np.mean(resized_roi[:, :, 1]))
        
        frame_count += 1
        
    # =========================
    # 🔥 IPPG PROCESSING
    # =========================

    lowcut = 0.7  
    highcut = 4.0  
    window_size = 10  
    poly_order = 2  

    R_channel = np.array(R_channel)
    G_channel = np.array(G_channel)
    B_channel = np.array(B_channel)
    green_signal = np.array(green_signal)

    # =========================
    # 🔥 IPPG2 (POS)
    # =========================
    X_POS = G_channel - 0.5 * (R_channel + B_channel)
    Y_POS = 3 * G_channel - 2 * R_channel - B_channel

    X_POS = (X_POS - np.mean(X_POS)) / np.std(X_POS)
    Y_POS = (Y_POS - np.mean(Y_POS)) / np.std(Y_POS)

    S_POS = X_POS + Y_POS

    filtered_signal_2 = apply_bandpass_filter(S_POS, lowcut, highcut, fps)
    smoothed_signal_2 = moving_average(filtered_signal_2, window_size)
    IPPG2 = remove_baseline(smoothed_signal_2, poly_order)
    IPPG2 = normalized_signal(IPPG2)

    # =========================
    # 🔥 IPPG1 (GREEN)
    # =========================
    filtered_signal_1 = apply_bandpass_filter(green_signal, lowcut, highcut, fps)
    smoothed_signal_1 = moving_average(filtered_signal_1, window_size)
    IPPG1 = remove_baseline(smoothed_signal_1, poly_order)
    IPPG1 = normalized_signal(IPPG1)

    # =========================
    # 🔥 IPPG3 (COMBINADO)
    # =========================
    min_len = min(len(IPPG1), len(IPPG2))
    IPPG1 = IPPG1[:min_len]
    IPPG2 = IPPG2[:min_len]

    IPPG3 = (IPPG1 + IPPG2) / 2
    IPPG3 = normalized_signal(IPPG3)

    # =========================
    # 🔥 SALVAR SINAIS COMPLETOS
    # =========================
    video_name = os.path.splitext(os.path.basename(input_video_path))[0]

    os.makedirs(output_base_dir, exist_ok=True)

    np.save(os.path.join(output_base_dir, f"{video_name}_ippg1.npy"), IPPG1)
    np.save(os.path.join(output_base_dir, f"{video_name}_ippg2.npy"), IPPG2)
    np.save(os.path.join(output_base_dir, f"{video_name}_ippg3.npy"), IPPG3)

    print(f"[OK] Sinais salvos em: {output_base_dir}")

    return True



In [1]:
import os

def main():

    base_dir = r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIPL-HR"

    output_dir = r"C:\Users\Douglas\Documents\VIPL_HR_IPPG"

    os.makedirs(output_dir, exist_ok=True)

    total_processados = 0
    total_pulados = 0
    total_erros = 0

    # ======================================================
    # PERCORRE p1-5, p6-10...
    # ======================================================
    for group in os.listdir(base_dir):

        group_path = os.path.join(base_dir, group)

        if not os.path.isdir(group_path):
            continue

        # ==================================================
        # p1, p2...
        # ==================================================
        for person in os.listdir(group_path):

            person_path = os.path.join(group_path, person)

            if not os.path.isdir(person_path):
                continue

            # ==================================================
            # v1 até v9
            # ==================================================
            for scenario in os.listdir(person_path):

                scenario_path = os.path.join(person_path, scenario)

                if (
                    not os.path.isdir(scenario_path)
                    or not scenario.startswith("v")
                ):
                    continue

                # ==================================================
                # source1, source2...
                # ==================================================
                for source in os.listdir(scenario_path):

                    source_path = os.path.join(scenario_path, source)

                    if not os.path.isdir(source_path):
                        continue

                    video_path = os.path.join(source_path, "video.avi")

                    if not os.path.exists(video_path):

                        print(f"[AVISO] Vídeo não encontrado:")
                        print(video_path)

                        continue

                    # ==================================================
                    # PASTA DE SAÍDA
                    # ==================================================
                    save_path = os.path.join(
                        output_dir,
                        group,
                        person,
                        scenario,
                        source
                    )

                    os.makedirs(save_path, exist_ok=True)

                    # ==================================================
                    # VERIFICAR SE JÁ EXISTEM OS 3 IPPGs
                    # ==================================================
                    ippg1_exists = any(
                        "ippg1" in f.lower()
                        for f in os.listdir(save_path)
                    )

                    ippg2_exists = any(
                        "ippg2" in f.lower()
                        for f in os.listdir(save_path)
                    )

                    ippg3_exists = any(
                        "ippg3" in f.lower()
                        for f in os.listdir(save_path)
                    )

                    # ==================================================
                    # SE TODOS EXISTEM → PULA
                    # ==================================================
                    if ippg1_exists and ippg2_exists and ippg3_exists:

                        print(f"[OK] Já processado:")
                        print(save_path)

                        total_pulados += 1

                        continue

                    # ==================================================
                    # PROCESSAR SOMENTE FALTANTES
                    # ==================================================
                    print(f"\n[PROCESSANDO]")
                    print(video_path)

                    try:

                        process_video(video_path, save_path)

                        total_processados += 1

                    except Exception as e:

                        print(f"[ERRO]")
                        print(video_path)
                        print(e)

                        total_erros += 1

    # ======================================================
    # RESUMO FINAL
    # ======================================================
    print("\n" + "=" * 50)
    print("FINALIZADO")
    print("=" * 50)

    print(f"Novos processados : {total_processados}")
    print(f"Já existentes     : {total_pulados}")
    print(f"Erros             : {total_erros}")


if __name__ == "__main__":
    main()

[OK] Já processado:
C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v1\source1
[OK] Já processado:
C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v1\source2
[OK] Já processado:
C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v1\source3

[PROCESSANDO]
C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIPL-HR\p1-5\p1\v1\source4\video.avi
[ERRO]
C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIPL-HR\p1-5\p1\v1\source4\video.avi
name 'process_video' is not defined
[OK] Já processado:
C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v2\source1
[OK] Já processado:
C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v2\source2
[OK] Já processado:
C:\Users\Douglas\Documents\VIPL_HR_IPPG\p1-5\p1\v2\source3

[PROCESSANDO]
C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIPL-HR\p1-5\p1\v2\source4\video.avi
[ERRO]
C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIP

In [ ]:
import os

# Script para verificar a estrutura do dataset e identificar quais pessoas estão completas ou incompletas em relação aos cenários e sources.

# ==========================================
# CAMINHO BASE
# ==========================================
base_dir = r"C:\Users\Douglas\Documents\VIPL_HR_IPPG"

# ==========================================
# CONTROLE
# ==========================================
persons_complete = []
persons_incomplete = []

# ==========================================
# VARREDURA
# ==========================================
for group in os.listdir(base_dir):

    group_path = os.path.join(base_dir, group)

    if not os.path.isdir(group_path):
        continue

    # p1, p2, p3...
    for person in os.listdir(group_path):

        person_path = os.path.join(group_path, person)

        if not os.path.isdir(person_path):
            continue

        missing = []

        # ==========================================
        # v1 até v9
        # ==========================================
        for scenario in range(1, 10):

            scenario_name = f"v{scenario}"
            scenario_path = os.path.join(person_path, scenario_name)

            # cenário não existe
            if not os.path.exists(scenario_path):
                missing.append(f"{scenario_name} (cenário ausente)")
                continue

            # ==========================================
            # source1 até source3
            # 🔥 IGNORA source4
            # ==========================================
            valid_sources = []

            for source in os.listdir(scenario_path):

                # ignora source4
                if source.lower() == "source4":
                    continue

                source_path = os.path.join(scenario_path, source)

                if os.path.isdir(source_path):
                    valid_sources.append(source)

            # cenário sem source válido
            if len(valid_sources) == 0:
                missing.append(f"{scenario_name} (sem sources válidos)")
                continue

            # ==========================================
            # VERIFICA IPPG
            # ==========================================
            for source in valid_sources:

                source_path = os.path.join(scenario_path, source)

                ippg_files = [
                    f for f in os.listdir(source_path)
                    if "ippg" in f.lower() and f.endswith(".npy")
                ]

                if len(ippg_files) == 0:
                    missing.append(f"{scenario_name}/{source}")

        # ==========================================
        # RESULTADO DA PESSOA
        # ==========================================
        if len(missing) == 0:
            persons_complete.append(person)
        else:
            persons_incomplete.append((person, missing))

# ==========================================
# RELATÓRIO
# ==========================================
print("\n==============================")
print("PESSOAS COMPLETAS")
print("==============================")

for p in sorted(persons_complete):
    print(p)

print(f"\nTOTAL COMPLETAS: {len(persons_complete)}")


print("\n==============================")
print("PESSOAS INCOMPLETAS")
print("==============================")

for person, missing in persons_incomplete:

    print(f"\n{person}")

    for item in missing:
        print(f"   -> Faltando: {item}")

print(f"\nTOTAL INCOMPLETAS: {len(persons_incomplete)}")

In [ ]:
import os

# ==========================================
# CAMINHOS
# ==========================================
base_video_dir = r"C:\Users\Douglas\OneDrive - cefet-rj.br\Documentos\Material _ faculdade\Projeto IC\VIPL-HR"

base_output_dir = r"C:\Users\Douglas\Documents\VIPL_HR_IPPG"

# ==========================================
# FUNÇÃO PRINCIPAL
# ==========================================
def process_missing_ippgs():

    total_missing = 0
    total_processed = 0

    # p1-5, p6-10...
    for group in os.listdir(base_video_dir):

        group_path = os.path.join(base_video_dir, group)

        if not os.path.isdir(group_path):
            continue

        # p1, p2...
        for person in os.listdir(group_path):

            person_path = os.path.join(group_path, person)

            if not os.path.isdir(person_path):
                continue

            # v1 até v9
            for scenario in os.listdir(person_path):

                scenario_path = os.path.join(person_path, scenario)

                if not os.path.isdir(scenario_path):
                    continue

                if not scenario.startswith("v"):
                    continue

                # source1 até source3
                for source in os.listdir(scenario_path):

                    # 🔥 IGNORAR SOURCE4
                    if source.lower() == "source4":
                        continue

                    source_path = os.path.join(scenario_path, source)

                    if not os.path.isdir(source_path):
                        continue

                    # ==========================================
                    # VERIFICAR SE JÁ EXISTE IPPG
                    # ==========================================
                    output_source_path = os.path.join(
                        base_output_dir,
                        group,
                        person,
                        scenario,
                        source
                    )

                    os.makedirs(output_source_path, exist_ok=True)

                    ippg_exists = False

                    if os.path.exists(output_source_path):

                        ippg_files = [
                            f for f in os.listdir(output_source_path)
                            if "ippg" in f.lower() and f.endswith(".npy")
                        ]

                        if len(ippg_files) > 0:
                            ippg_exists = True

                    # ==========================================
                    # SE NÃO EXISTE → PROCESSAR
                    # ==========================================
                    if not ippg_exists:

                        total_missing += 1

                        video_path = os.path.join(source_path, "video.avi")

                        if not os.path.exists(video_path):

                            print(f"[VÍDEO NÃO ENCONTRADO]")
                            print(video_path)
                            continue

                        print("\n===================================")
                        print("[GERANDO IPPG]")
                        print(video_path)
                        print("===================================")

                        try:
                            # 🔥 SUA FUNÇÃO
                            process_video(video_path, output_source_path)

                            total_processed += 1

                        except Exception as e:

                            print(f"[ERRO]")
                            print(video_path)
                            print(e)

    print("\n===================================")
    print("FINALIZADO")
    print("===================================")
    print(f"IPPGs faltantes encontrados : {total_missing}")
    print(f"IPPGs gerados com sucesso   : {total_processed}")


# ==========================================
# EXECUÇÃO
# ==========================================
if __name__ == "__main__":

    process_missing_ippgs()

In [4]:
import numpy as np

# Carregar arquivo .npy
data = np.load(r"C:\Users\Douglas\Documents\VIPL_IPPG_TRAIN\p1-5\p5\v1\source1\video_ippg2.npy")

# Mostrar dimensões
print("Shape:", data.shape)

# Número de dimensões
print("Número de dimensões:", data.ndim)

# Tipo dos dados
print("Tipo:", data.dtype)

# Quantidade total de elementos
print("Total de elementos:", data.size)

Shape: (762,)
Número de dimensões: 1
Tipo: float64
Total de elementos: 762
